Configuration
=============

Beam Corset's visualization functions offer quite a few customization options. In most scenarios, the default settings should be sufficient. However, if you want to overwrite them for one call, you will most likely want to adjust them for all calls. This can be facilitated using the :class:`~corset.config.Config` class and its inner classes that hold global configuration options as class variables.

The available configuration options are listed in the :mod:`corset.config` module documentation.

Plotting Functions
------------------

For example, for optical setup plots and related plots, we can specify whether we want to show a legend or not or which units to use for which quantities.

In [ ]:
from corset import Beam

beam = Beam.from_gauss(0, 100e-6, 1064e-9)
beam.plot(show_legend=True); # show the legend

If no value is passed, as we would typically do in most situations, the value for `show_legend` is taken from the global configuration value :attr:`Config.PlotSetup.show_legend <corset.config.Config.PlotSetup.show_legend>`. Using the config, we can set this value without having to specify it at each call. This also means we can specify it in cases where we can not explicitly pass arguments like the implicit plotting done for PNG representations in Jupyter notebooks.

In [ ]:
from corset import Config

Config.PlotSetup.show_legend = True
beam # implicit Beam.plot() call uses config values

One common use case where this is also very practical is to explicitly set the ranges of the reachability and sensitivity plots so they share the same scale across all solutions. In this case we do not specify the ranges in the :attr:`Config.PlotReachability <corset.config.Config.PlotReachability>` and :attr:`Config.PlotSensitivity <corset.config.Config.PlotSensitivity>` inner classes, because they interpret no values as using automatic range instead of resorting to configuration values. Instead, we set the :attr:`Config.PlotAll.reachability_kwargs <corset.config.Config.PlotAll.reachability_kwargs>` and :attr:`Config.PlotAll.sensitivity_kwargs <corset.config.Config.PlotAll.sensitivity_kwargs>` dictionaries which :meth:`ModeMatchingSolution.plot_all() <corset.solver.ModeMatchingSolution.plot_all>` uses to pass arguments to the respective plotting functions.

In [ ]:
from corset import ThinLens, ShiftingRange, mode_match

solutions = mode_match(Beam.from_gauss(0.0, 500e-6, 1064e-9), Beam.from_gauss(1.0, 100e-6, 1064e-9), [ShiftingRange(0.0, 0.8)], [ThinLens(f) for f in [100e-3, 150e-3]], 2, 2)

# fix ranges so they are the same for all plots
Config.PlotAll.reachability_kwargs = {"focus_range": (-20e-3, 20e-3), "waist_range": (90e-6, 110e-6)}
Config.PlotAll.sensitivity_kwargs = {"x_displacement": 10e-3, "y_displacement": 20e-3}
solutions.display_all()

Customizing Tables
------------------

We can also use the config to specify which columns to show in the various `~pandas.DataFrame` based representations.

In [ ]:
Config.Repr.solution_summary_columns = ["overlap", "elements", "max_sensitivity", "min_coupling"]
solutions

Units
-----

While Beam Corset uses SI basis units internally, the plots and tables are displayed using units that are more appropriate for the respective quantities. The units to use can be configured in groups using the four fundamental different types of quantities used in the plots and tables:

- Axial distances along the optical axis for positions of elements and beam waists.
- Radial distances perpendicular to the optical axis for the beams radial profile.
- Fractions for the mode overlap of a solution and the coupling between degree of freedom.
- Sensitivities for the sensitivities and cross sensitivities of elements in a solution.

The are specified in the :class:`~corset.config.Config.Units` section of the config using the constants from :class:`~corset.display.Units` which also has common shorthand aliases for the units.

In [ ]:
from corset import Units

Config.Units.axial = Units.Length.CENTIMETER # fully qualified name
Config.Units.fraction = Units.ul # shorthand for Units.Fraction.UNITY

solutions[0]

It is also possible to create custom units:

In [ ]:
from corset.display import LengthUnit

DECIMETER = LengthUnit(unicode="dm", tex="\\mathrm{dm}", factor=0.1, decimals=2)
Config.Units.axial = DECIMETER

solutions[0]